In [24]:
from pennylane import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb
import qaoa_aux 
# import importlib
# importlib.reload(qaoa_aux)

<module 'qaoa_aux' from '/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning/extension/qaoa_aux.py'>

In [2]:
def U_B(beta,edge_list):
    for edge_idx, (u, v) in enumerate(edge_list):
        theta = beta[edge_idx]
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * theta,wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [3]:
def U_C(gamma,cost_terms):
    # we have term index + cost terms so all the even numbered ZZ's
    for term_idx, (_, wires, coeff) in enumerate(cost_terms):
        theta = 2 * gamma[term_idx] * coeff
        # example for 4 consectuve z's
        #qml.PauliRot(theta, "ZZZZ", wires=[0,1,2,3])
        # pennylane will decompose this into CNOT+RZ ladder 
        qml.PauliRot(theta,"Z" * len(wires),wires=list(wires))

In [4]:
def objective(gammas,betas,circuit,hyperedge_list,cost_terms,weights):
    expectation_values = circuit(gammas, betas)
    value = 0.0

    # weighted constant term for each hyperedge
    for edge_idx, edge in enumerate(hyperedge_list):
        k = len(edge)
        #w_e * 1 - (1/2^k-1)
        value += weights[edge_idx] * (1 - 1 / (2 ** (k - 1)))

    # coeff is weighted in build_cost_terms()
    for term_idx, (_, _, coeff) in enumerate(cost_terms):
        value += coeff * expectation_values[term_idx]

    return value

In [6]:

def run_qaoa(
    hyperedge_list,
    n,
    weights,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires=n
    assert len(weights)==len(hyperedge_list)
    # Build mixer edges and cost terms
    edge_list = qaoa_aux.clique_expansion(hyperedge_list)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

    # Balanced initial state
    target_weight = n_wires // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires,target_weight)

    dev = qml.device("lightning.qubit",wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    # Random parameter initialization
    rng = np.random.default_rng(seed)
    beta_params = np.array(scaling_constant * rng.random((len(edge_list), num_layers)),requires_grad=True)
    gamma_params = np.array(scaling_constant * rng.random((len(cost_terms), num_layers)),requires_grad=True)

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    initial_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
    objective_values.append(initial_value)

    print(f"Initial cut = {initial_value:.6f}")

    # Optimization
    for trial in range(max_trials):

        gamma_params, beta_params = opt.step(
            lambda g, b: -objective(
                g,
                b,
                circuit,
                hyperedge_list,
                cost_terms,
                weights
            ),
            gamma_params,
            beta_params
        )

        current_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
        objective_values.append(current_value)

        if trial % 10 == 0:
            print(f"Iteration {trial:3d} | "f"cut = {current_value:.6f}")

    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "edge_list": edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

In [7]:
P = np.load("../data/poisson_16_12_3/P/03.npy")
w= np.load("../data/poisson_16_12_3/w/03.npy")

hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)

In [9]:

result = run_qaoa(
    hyperedge_list,
    weights=w,
    n=12,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.05,
    max_trials=300
)

Initial cut = 12.607519
Iteration   0 | cut = 12.762909
Iteration  10 | cut = 14.188213
Iteration  20 | cut = 14.686550
Iteration  30 | cut = 14.872401
Iteration  40 | cut = 14.993464
Iteration  50 | cut = 15.071466
Iteration  60 | cut = 15.157045
Iteration  70 | cut = 15.203829
Iteration  80 | cut = 15.230109
Iteration  90 | cut = 15.261139
Iteration 100 | cut = 15.288187
Iteration 110 | cut = 15.308590
Iteration 120 | cut = 15.328176
Iteration 130 | cut = 15.350026
Iteration 140 | cut = 15.376969
Iteration 150 | cut = 15.391462
Iteration 160 | cut = 15.395881
Iteration 170 | cut = 15.398634
Iteration 180 | cut = 15.400146
Iteration 190 | cut = 15.400837
Iteration 200 | cut = 15.401363
Iteration 210 | cut = 15.401647
Iteration 220 | cut = 15.401784
Iteration 230 | cut = 15.401864
Iteration 240 | cut = 15.401928
Iteration 250 | cut = 15.401996
Iteration 260 | cut = 15.402072
Iteration 270 | cut = 15.402156
Iteration 280 | cut = 15.402249
Iteration 290 | cut = 15.402352


In [14]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_3/P"
folder_w = "../data/poisson_16_12_3/w"
p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_max_bisection = []

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p}")

        result = run_qaoa(hyperedge_list,weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=200)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results_max_bisection.append({
            "dataset": "poisson_16_12_3",
            "objective":"max_bisection",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })



Running instance 01 | p=3
Initial cut = 12.928743
Iteration   0 | cut = 13.085121
Iteration  10 | cut = 13.756571
Iteration  20 | cut = 14.449345
Iteration  30 | cut = 14.883552
Iteration  40 | cut = 15.285958
Iteration  50 | cut = 15.562892
Iteration  60 | cut = 15.725570
Iteration  70 | cut = 15.817176
Iteration  80 | cut = 15.875190
Iteration  90 | cut = 15.907053
Iteration 100 | cut = 15.927094
Iteration 110 | cut = 15.940324
Iteration 120 | cut = 15.949651
Iteration 130 | cut = 15.956236
Iteration 140 | cut = 15.961088
Iteration 150 | cut = 15.964879
Iteration 160 | cut = 15.967984
Iteration 170 | cut = 15.970577
Iteration 180 | cut = 15.972769
Iteration 190 | cut = 15.974639

Running instance 02 | p=3
Initial cut = 11.483656
Iteration   0 | cut = 11.632298
Iteration  10 | cut = 12.190995
Iteration  20 | cut = 13.373622
Iteration  30 | cut = 14.120411
Iteration  40 | cut = 14.590322
Iteration  50 | cut = 14.969513
Iteration  60 | cut = 15.230826
Iteration  70 | cut = 15.398736
It

In [16]:
import pandas as pd
df = pd.DataFrame(results_max_bisection)
csv_path = "poisson_16_12_3_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_3_qaoa_max_bisection.csv


In [25]:
def run_qaoa_modified(
    hyperedge_list,
    n,
    weights,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires=n
    assert len(weights)==len(hyperedge_list)
    # Build mixer edges and cost terms
    edge_list = qaoa_aux.connected_clique_expansion(hyperedge_list,n)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

    # Balanced initial state
    target_weight = n_wires // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires,target_weight)

    dev = qml.device("lightning.qubit",wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    # Random parameter initialization
    rng = np.random.default_rng(seed)
    beta_params = np.array(scaling_constant * rng.random((len(edge_list), num_layers)),requires_grad=True)
    gamma_params = np.array(scaling_constant * rng.random((len(cost_terms), num_layers)),requires_grad=True)

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    initial_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
    objective_values.append(initial_value)

    print(f"Initial cut = {initial_value:.6f}")

    # Optimization
    for trial in range(max_trials):

        gamma_params, beta_params = opt.step(
            lambda g, b: -objective(
                g,
                b,
                circuit,
                hyperedge_list,
                cost_terms,
                weights
            ),
            gamma_params,
            beta_params
        )

        current_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
        objective_values.append(current_value)

        if trial % 10 == 0:
            print(f"Iteration {trial:3d} | "f"cut = {current_value:.6f}")

    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "edge_list": edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

In [26]:
import os
import glob
import pandas as pd

def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges
    
folder = "/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning/data/congress-bills/numpy_20"
p_files = sorted(glob.glob(f"{folder}/*.npy"))

results_max_bisection_congress = []

stop=0

for p_file in p_files:
    if stop>9: break
    P = np.load(p_file)
    instance = os.path.splitext(os.path.basename(p_file))[0]

    edges = get_edge_list(P)
    w = np.ones(len(edges))
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p}")

        result = run_qaoa_modified(hyperedge_list,weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results_max_bisection_congress.append({
            "dataset": "congress",
            "objective":"max_bisection",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })
    stop+=1



Running instance congress-bills_01 | p=3
Initial cut = 1.000000
Iteration   0 | cut = 1.000000
Iteration  10 | cut = 1.000000
Iteration  20 | cut = 1.000000
Iteration  30 | cut = 1.000000
Iteration  40 | cut = 1.000000
Iteration  50 | cut = 1.000000
Iteration  60 | cut = 1.000000
Iteration  70 | cut = 1.000000
Iteration  80 | cut = 1.000000
Iteration  90 | cut = 1.000000
Iteration 100 | cut = 1.000000
Iteration 110 | cut = 1.000000
Iteration 120 | cut = 1.000000
Iteration 130 | cut = 1.000000
Iteration 140 | cut = 1.000000
Iteration 150 | cut = 1.000000
Iteration 160 | cut = 1.000000
Iteration 170 | cut = 1.000000
Iteration 180 | cut = 1.000000
Iteration 190 | cut = 1.000000
Iteration 200 | cut = 1.000000
Iteration 210 | cut = 1.000000
Iteration 220 | cut = 1.000000
Iteration 230 | cut = 1.000000
Iteration 240 | cut = 1.000000
Iteration 250 | cut = 1.000000
Iteration 260 | cut = 1.000000
Iteration 270 | cut = 1.000000
Iteration 280 | cut = 1.000000
Iteration 290 | cut = 1.000000

Runn

In [27]:
import pandas as pd
df = pd.DataFrame(results_max_bisection_congress)
csv_path = "congress_qaoa_max_bisection.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: congress_qaoa_max_bisection.csv
